In [11]:
import os, sys
from bertviz import head_view, model_view
from transformers import RobertaTokenizer, RobertaConfig
from safetensors import safe_open
import torch

sys.path.append("../")
from utils.parameters import *
from train_two_tower_model.two_tower_models import DualTowerForSequenceClassification_visual

In [12]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

config = RobertaConfig(
    vocab_size_1=9,
    vocab_size_2=10_000,
    hidden_size=256, # embedding size
    max_position_embeddings=128,
    num_decoder_layers=3,
    num_attention_heads=8,
    intermediate_size=3072,
    type_vocab_size=1,
    pad_token_id=3,
    bos_token_id=0,
    eos_token_id=1,
    position_embedding_type="relative_key",
)
# * tower model
tower_model = DualTowerForSequenceClassification_visual(config, is_pretrained=True, pretrained_version=5)
tensors = {}
with safe_open(SAVED_MODEL_DIR_SEMI_V2+"/model.safetensors", framework="pt") as f:
    for k in f.keys():
        tensors[k] = f.get_tensor(k)
tower_model.load_state_dict(tensors)
tower_model.to(device)
tower_model.eval()

tokenizer1 = RobertaTokenizer.from_pretrained(TOKEN_DIR_BASE)
tokenizer2 = RobertaTokenizer.from_pretrained(TOKEN_DIR_BPE)

Pretrained weights loaded successfully. (v5)


In [13]:
import heapq

def gen_candidate_seqs():
    hp = list()
    i = 0
    # * get the top 10 prediction probabilities sequences
    with open(os.path.join(SAMPLES_PATH, "distinct_positive_samples_HEK293T.csv"), "r") as f:
        while line := f.readline().strip():
            inputs1 = tokenizer1(line, return_tensors="pt")
            inputs2 = tokenizer2(line, return_tensors="pt")
            inputs = {"input_ids_1": inputs1["input_ids"].to(device), "input_ids_2": inputs2["input_ids"].to(device)}
            outputs = tower_model(**inputs)
            heapq.heappush(hp, (-outputs.logits[0][1].item(), line))
            i += 1
            if i % 2048 == 0:
                print(i)

    top10 = heapq.nsmallest(10, hp)
    print(top10)

# gen_candidate_seqs()

In [14]:
# candidate_seqs = [(4.247534275054932, 'CCCTGTCCTACCCCGCCCCTCGTCCTGGCCCCTCCCCTGCCCC'),
#                   (4.206146717071533, 'CCCTGTAGGGACAGCCCACTTCCAGGCCCCGCCCTTGATCCC'),
#                   (4.186415195465088, 'CCCGGCCTCTCGCCCCGCCCCCATGTCCCGCCC'),
#                   (4.184805393218994, 'CCCTTCCCGAGTCCTCCCCGAGCCCTCCCCGAGCCCTTCCCC'),
#                   (4.172497749328613, 'CCCTGCCCCCGCCCTTGCTCCGCCCCGCCC'),
#                   (4.171431064605713, 'CCCGAGTCCTCCCCGAGCCCTCCCCGAGCCCTTCCCC'),
#                   (4.1703386306762695, 'CCCCAGGCCCCGCCCCTCGTGCTAGCCCCGCCCC'),
#                   (4.165041923522949, 'CCCATCCTAGGCCCCGCCCATCCCAGACACCGGCCC'),
#                   (4.162461757659912, 'CCCCCACCCTCCGCGCCCGCCCGCTCTTTCCCCGCCCC'),
#                   (4.155589580535889, 'CCCACCCCTCCCACCGCCTACAGCCCGGCCCCGCCCC')]

candidate_seqs2 = [(-4.281908988952637, 'CCCCCCCGCGCGGCCCCGCCCCC'),
                  (-4.243957996368408, 'CCCCCCCGCCCCCGCCACCCCGCACCCCGCCCCCCGGCATGGCTTTCCCC'),
                  (-4.230985641479492, 'CCCCGCCCATCTTCCAGGCCCCGCCC'),
                  (-4.230202674865723, 'CCCCCGGCCCGGCCTCAGGCCCCGCCC'),
                  (-4.230099678039551, 'CCCCGCCTCCCGTCGGCCCGCCCCCAAACCCCGCCC'),
                  (-4.218786716461182, 'CCCCACCCCTTACCGCCTGGCCCCGCCCACGCCCC'),
                  (-4.208957195281982, 'CCCCACCCTCCTTCGGGCCCCGCCCCTCCCCTCCCC'),
                  (-4.207056999206543, 'CCCGCCCCGCCCCGCCCCTTGGCCTGAAGACCCCGCCC'),
                  (-4.196073055267334, 'CCCTCATAGGCCACGCCCCGGACCCGCCCCCC'),
                  (-4.193173408508301, 'CCCGCCCAGGACCTAGCCCCGCCC')]

# sentence_a = "CCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCC"
# sentence_a = "CCCGCTGTACCCTGCGCCCTCGCCC"
# sentence_a = candidate_seqs[4][1]
sentence_a = candidate_seqs2[3][1]

# inputs1 = tokenizer1.encode_plus(sentence_a, return_tensors='pt', padding='max_length', max_length=MAX_LEN_BASE)
# inputs2 = tokenizer2.encode_plus(sentence_a, return_tensors='pt', padding='max_length', max_length=MAX_LEN_BPE)
inputs1 = tokenizer1.encode_plus(sentence_a, return_tensors='pt')
inputs2 = tokenizer2.encode_plus(sentence_a, return_tensors='pt')
inputs = {'input_ids_1': inputs1['input_ids'].to(device), 'input_ids_2': inputs2['input_ids'].to(device)}
print(inputs)

# Get the model's attention weights
outputs = tower_model(**inputs)
print(outputs.logits)

base_self_attention = list()
bpe_self_attention = list()
for i, att in enumerate(outputs.attentions):
    if i % 2 == 0:
        base_self_attention.append(att)
    else:
        bpe_self_attention.append(att)

base_cross_attention = list()
bpe_cross_attention = list()
for i, att in enumerate(outputs.cross_attentions):
    if i % 2 == 0:
        base_cross_attention.append(att)
    else:
        bpe_cross_attention.append(att)

# Use BertViz to visualize the attention
tokens1 = tokenizer1.convert_ids_to_tokens(inputs['input_ids_1'][0])
tokens2 = tokenizer2.convert_ids_to_tokens(inputs['input_ids_2'][0])
print(tokens1)
print(tokens2)

{'input_ids_1': tensor([[0, 7, 7, 7, 7, 7, 8, 8, 7, 7, 7, 8, 8, 7, 7, 6, 7, 5, 8, 8, 7, 7, 7, 7,
         8, 7, 7, 7, 1]], device='cuda:0'), 'input_ids_2': tensor([[   0,    7,   23,    5, 2011,    6,   20,    5,    1]],
       device='cuda:0')}
tensor([[-4.0078,  3.8081]], device='cuda:0', grad_fn=<AddmmBackward0>)
['<s>', 'C', 'C', 'C', 'C', 'C', 'G', 'G', 'C', 'C', 'C', 'G', 'G', 'C', 'C', 'T', 'C', 'A', 'G', 'G', 'C', 'C', 'C', 'C', 'G', 'C', 'C', 'C', '</s>']
['<s>', 'CCCCC', 'GG', 'CCC', 'GGCCTCAGG', 'CCCC', 'G', 'CCC', '</s>']


In [15]:
head_view(base_self_attention, tokens1)

<IPython.core.display.Javascript object>

In [16]:
head_view(bpe_self_attention, tokens2)

<IPython.core.display.Javascript object>

In [17]:
head_view(cross_attention=base_cross_attention, encoder_tokens=tokens2, decoder_tokens=tokens1)

<IPython.core.display.Javascript object>

In [18]:
head_view(cross_attention=bpe_cross_attention, encoder_tokens=tokens1, decoder_tokens=tokens2)

<IPython.core.display.Javascript object>

In [19]:
# Use BertViz to visualize the attention with model_view
# model_view(attention2, tokens2)